# NSMC 영화 리뷰 감성 분류 (KcBERT 파인튜닝)

네이버 영화 리뷰(NSMC) 15만 건으로 한국어 BERT(`beomi/kcbert-base`)를 파인튜닝해 긍정/부정을 분류합니다.

- **학습 루프 직접 구현**: PyTorch Lightning이나 HF `Trainer` 없이 순전파 → 손실 → 역전파 → 업데이트를 직접 작성
- **검증 데이터 분리**: train의 10%를 validation으로 떼고, test는 마지막에 한 번만 평가
- **혼합 정밀도(AMP)** 와 **선형 warmup 스케줄러** 적용

| 데이터 | Loss | Accuracy |
|---|---|---|
| Validation (15,000) | 0.2359 | **90.16%** |
| Test (49,997) | 0.2444 | **89.92%** |

1 epoch, Colab T4 GPU · torch 2.11 · transformers 5.22

『Do it! 자연어 처리』 4장 문서 분류 실습을 참고했으며, 더 이상 설치되지 않는 `ratsnlp` 대신 최신 라이브러리로 다시 구현했습니다.

## 0. 환경 확인
Colab 기본 환경(`torch`, `transformers`, `pandas`)만 사용합니다. 원 교재의 `ratsnlp`는 옛 버전 의존성을 고정해 최신 Python에서 설치되지 않아 사용하지 않습니다.

In [2]:
# transformers가 없다는 에러가 나면 아래 줄의 주석을 풀고 실행하세요.
# !pip install -q transformers

import torch          # PyTorch
import transformers   # Hugging Face 사전학습 모델 라이브러리 (PyTorch 위에서 동작)

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("GPU 사용 가능:", torch.cuda.is_available())  # False면 런타임 유형을 GPU로 바꾸세요

torch: 2.11.0+cu130
transformers: 5.22.0
GPU 사용 가능: True


## 1. 설정값
학습에 쓰는 숫자들을 한곳에 모읍니다. 책의 `ClassificationTrainArguments`와 같은 역할입니다.

In [3]:
from dataclasses import dataclass

@dataclass
class Config:  # 하이퍼파라미터를 한곳에서 관리 (이름: 타입 = 기본값)
    model_name: str = "beomi/kcbert-base"  # 책과 같은 사전학습 모델
    max_length: int = 128                  # 문장을 최대 몇 토큰까지 볼지
    batch_size: int = 32                   # 한 번에 학습하는 문장 수
    learning_rate: float = 5e-5            # 파인튜닝에서 흔히 쓰는 작은 학습률
    epochs: int = 1                        # 책은 3. 1에폭만 해도 정확도가 충분히 나옵니다
    warmup_ratio: float = 0.1              # 처음 10% 스텝 동안 학습률을 0에서 천천히 올림
    val_ratio: float = 0.1                 # 학습 데이터 중 검증용으로 떼어 둘 비율
    train_size: int | None = None          # 빨리 돌려 보고 싶으면 20000 같은 숫자로 (None = 전체)
    seed: int = 42
    save_dir: str = "nsmc-kcbert"

cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# GPU를 쓸 수 있으면 cuda, 아니면 cpu

# AMP(혼합 정밀도): GPU에서 일부 계산을 float16으로 해서 속도를 올리고 메모리를 아낍니다.
use_amp = device.type == "cuda"
print(cfg)
print("device:", device)

Config(model_name='beomi/kcbert-base', max_length=128, batch_size=32, learning_rate=5e-05, epochs=1, warmup_ratio=0.1, val_ratio=0.1, train_size=None, seed=42, save_dir='nsmc-kcbert')
device: cuda


## 2. 랜덤 시드 고정
시드를 고정하면 다시 실행해도 데이터 섞는 순서, 가중치 초기값 등이 같아져서 결과를 재현할 수 있습니다.

In [4]:
import random
import numpy as np

def set_seed(seed):  # 재현성을 위해 모든 난수 생성기의 시드 고정
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

## 3. 말뭉치 다운로드 및 전처리

NSMC 원본 파일(탭으로 구분된 `id`, `document`, `label`)을 GitHub에서 바로 읽습니다.
- `label`: 0 = 부정, 1 = 긍정
- 리뷰 안에 따옴표(`"`)가 든 줄이 있어서 `quoting=csv.QUOTE_NONE`으로 따옴표를 특별 취급하지 않게 합니다.
- 내용이 빈 리뷰가 몇 개 있어서 제거합니다.

In [5]:
import csv
import pandas as pd

BASE_URL = "https://raw.githubusercontent.com/e9t/nsmc/master/"  # NSMC 원본 저장소
# 탭 구분 파일. 리뷰 속 따옴표(")가 칸 묶음 기호로 해석되지 않도록 QUOTE_NONE
train_df = pd.read_csv(BASE_URL + "ratings_train.txt", sep="\t", quoting=csv.QUOTE_NONE)
test_df = pd.read_csv(BASE_URL + "ratings_test.txt", sep="\t", quoting=csv.QUOTE_NONE)

train_df = train_df.dropna(subset=["document"]).reset_index(drop=True)  # 빈 리뷰 제거 후 인덱스 재정렬
test_df = test_df.dropna(subset=["document"]).reset_index(drop=True)

print("train:", len(train_df), "/ test:", len(test_df))
train_df.head()

train: 149995 / test: 49997


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [6]:
# 데이터 살펴보기: 레이블 비율과 리뷰 길이
print(train_df["label"].value_counts())
print(train_df["document"].str.len().describe())  # 글자 수 통계 → max_length 설정 근거

label
0    75170
1    74825
Name: count, dtype: int64
count    149995.000000
mean         35.238568
std          29.582269
min           1.000000
25%          16.000000
50%          27.000000
75%          42.000000
max         158.000000
Name: document, dtype: float64


### 학습 / 검증 분리
NSMC에는 train과 test만 있습니다. test는 **마지막 평가에만** 쓰고, 학습 중에 성능을 확인할 검증(validation) 데이터는 train에서 떼어 냅니다. (원 교재는 test를 검증에 바로 썼지만, test를 보면서 설정을 고르면 test 성능이 실제보다 낙관적으로 나오므로 분리했습니다.)

In [7]:
if cfg.train_size is not None:
    train_df = train_df.sample(n=cfg.train_size, random_state=cfg.seed).reset_index(drop=True)

val_df = train_df.sample(frac=cfg.val_ratio, random_state=cfg.seed)  # 10% 무작위 추출 (시드 고정)
train_df = train_df.drop(val_df.index).reset_index(drop=True)  # 검증에 뽑힌 줄은 학습에서 제외
val_df = val_df.reset_index(drop=True)

print("train:", len(train_df), "/ val:", len(val_df), "/ test:", len(test_df))

train: 134995 / val: 15000 / test: 49997


## 4. 토크나이저
문장을 모델이 이해하는 숫자(토큰 ID)로 바꿉니다. KcBERT는 WordPiece 토크나이저를 쓰며, 단어 중간 조각에는 `##`이 붙습니다.
- `input_ids`: 각 토큰의 번호. 맨 앞 `[CLS]`, 맨 뒤 `[SEP]`가 자동으로 붙습니다.
- `token_type_ids`: 문장 A/B 구분용 (문장이 하나라 전부 0)
- `attention_mask`: 진짜 토큰은 1, 패딩은 0

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

sample = train_df["document"].iloc[0]  # 예시 문장 하나
encoded = tokenizer(sample)
print("원문:", sample)
print("토큰:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print(encoded)

원문: 흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나
토큰: ['[CLS]', '흠', '.', '.', '.', '포', '##스터', '##보고', '초딩', '##영화', '##줄', '.', '.', '.', '.', '오버', '##연기', '##조차', '가볍', '##지', '않', '##구나', '[SEP]']
{'input_ids': [2, 3521, 17, 17, 17, 3294, 13069, 8190, 10635, 13796, 4006, 17, 17, 17, 17, 17613, 19625, 9790, 17775, 4102, 2175, 8030, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


## 5. 데이터셋과 데이터로더
- `Dataset`: "i번째 샘플을 달라"고 하면 하나를 돌려주는 클래스 (`__len__`, `__getitem__` 두 개만 있으면 됨)
- `DataLoader`: Dataset에서 샘플을 `batch_size`개씩 묶고, 학습 데이터는 매 에폭 섞어 줌

여기서는 모든 문장을 처음에 한 번에 토큰화하고, 길이를 `max_length`로 맞춥니다(`padding="max_length"`). 책과 같은 방식이고 이해하기 쉽지만, 짧은 문장에도 패딩이 많이 붙어 계산이 낭비됩니다. (개선 여지: 배치별 동적 패딩)

In [9]:
from torch.utils.data import Dataset, DataLoader

class NSMCDataset(Dataset):  # train / val / test 모두 이 클래스 하나로 처리
    def __init__(self, df, tokenizer, max_length):
        self.encodings = tokenizer(
            df["document"].tolist(),  # 토크나이저 입력용 리스트
            truncation=True,          # max_length보다 길면 자르기
            max_length=max_length,
            padding="max_length",     # 짧으면 [PAD]로 채워서 길이 맞추기
        )
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # dataset[idx]로 샘플 하나를 꺼낼 때 호출됨 (DataLoader가 배치 크기만큼 호출해 묶음)
        item = {key: torch.tensor(values[idx]) for key, values in self.encodings.items()}  # idx번째 토큰 정보 → 텐서
        item["labels"] = torch.tensor(self.labels[idx])  # 정답 레이블 추가
        return item

train_dataset = NSMCDataset(train_df, tokenizer, cfg.max_length)
val_dataset = NSMCDataset(val_df, tokenizer, cfg.max_length)
test_dataset = NSMCDataset(test_df, tokenizer, cfg.max_length)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size * 2, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size * 2, shuffle=False, num_workers=2)

# 배치 하나를 꺼내서 shape 확인: (batch_size, max_length)
batch = next(iter(train_loader))
for key, value in batch.items():
    print(key, tuple(value.shape))

input_ids (32, 128)
token_type_ids (32, 128)
attention_mask (32, 128)
labels (32,)


## 6. 모델 초기화
사전학습된 KcBERT 위에 **분류용 층(classifier)** 을 하나 얹은 모델을 불러옵니다.

실행하면 `Some weights ... were not initialized ... classifier` 같은 경고가 뜨는데 **정상**입니다. BERT 본체는 사전학습된 가중치를 쓰고, 새로 얹은 분류 층만 무작위로 초기화됐다는 뜻이에요. 이 분류 층과 본체를 함께 학습시키는 것이 파인튜닝입니다.

In [10]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(cfg.model_name, num_labels=2)
model.to(device)  # 모든 가중치를 GPU 메모리로 이동

num_params = sum(p.numel() for p in model.parameters())
print(f"파라미터 수: {num_params / 1e6:.1f}M")

파라미터 수: 108.9M


## 7. 옵티마이저와 학습률 스케줄러
- **AdamW**: Transformer 파인튜닝의 표준 옵티마이저 (`torch.optim`에 있는 것을 씁니다. 옛 코드의 `transformers.AdamW`는 삭제됨)
- **선형 warmup 스케줄러**: 학습률을 처음엔 0에서 천천히 올렸다가(warmup), 이후 끝까지 선형으로 줄입니다. 처음부터 큰 학습률로 사전학습 가중치가 망가지는 것을 막아 줍니다.

In [11]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(), lr=cfg.learning_rate)

total_steps = len(train_loader) * cfg.epochs
warmup_steps = int(total_steps * cfg.warmup_ratio)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

print("전체 스텝:", total_steps, "/ warmup 스텝:", warmup_steps)

전체 스텝: 4219 / warmup 스텝: 421


## 8. 평가 함수
검증·테스트 데이터에서 손실과 정확도를 계산합니다.
- `model.eval()`: 드롭아웃을 끄는 평가 모드 (끝나면 `model.train()`으로 되돌림)
- `@torch.no_grad()`: 기울기를 계산하지 않아 메모리와 시간을 아낌

In [12]:
import torch.nn.functional as F

@torch.no_grad()
def evaluate(model, loader):
    model.eval()  # 드롭아웃 끄기
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        labels = batch.pop("labels")
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(**batch).logits
            loss = F.cross_entropy(logits, labels)
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=-1) == labels).sum().item()
        total += labels.size(0)
    model.train()
    return total_loss / total, correct / total  # (평균 손실, 정확도)

## 9. 학습 루프 (직접 구현)
원 교재에서는 PyTorch Lightning이 감추던 부분을 직접 작성했습니다. 한 스텝은 다섯 단계입니다.
1. `optimizer.zero_grad()`: 이전 스텝의 기울기 지우기 (안 지우면 계속 더해짐)
2. 순전파: `model(**batch)`로 예측값(logits) 계산
3. 손실 계산: 예측과 정답의 차이 (`cross_entropy`)
4. `loss.backward()`: 각 가중치의 기울기 계산
5. `optimizer.step()`: 기울기 방향으로 가중치 업데이트 (+ `scheduler.step()`으로 학습률 조정)

`scaler`는 AMP용 도구입니다. float16은 아주 작은 기울기가 0이 되어 버릴 수 있어서, 손실을 크게 키워서 `backward` 한 뒤 다시 줄여 줍니다. AMP를 빼면 `scaler.scale(loss).backward()` → `loss.backward()`, `scaler.step(optimizer)` → `optimizer.step()`이 됩니다.

T4 GPU 기준 전체 데이터 1 epoch(4,219 step)를 학습했습니다.

In [13]:
from tqdm.auto import tqdm

scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
model.train()

for epoch in range(cfg.epochs):
    progress = tqdm(train_loader, desc=f"epoch {epoch + 1}/{cfg.epochs}")
    for step, batch in enumerate(progress):
        batch = {key: value.to(device) for key, value in batch.items()}
        labels = batch.pop("labels")

        optimizer.zero_grad()                                   # 1. 이전 스텝의 기울기 초기화
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(**batch).logits                      # 2. 순전파 (예측값 계산)
            loss = F.cross_entropy(logits, labels)              # 3. 손실 계산
        scaler.scale(loss).backward()                           # 4. 역전파 (기울기 계산)
        scaler.step(optimizer)                                  # 5. 가중치 업데이트
        scaler.update()
        scheduler.step()

        if step % 50 == 0:
            progress.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.2e}")

    val_loss, val_acc = evaluate(model, val_loader)
    print(f"[epoch {epoch + 1}] val_loss={val_loss:.4f}  val_acc={val_acc:.4f}")

[epoch 1] val_loss=0.2359  val_acc=0.9016


### 최종 테스트 평가와 저장
학습이 끝난 뒤 test로 한 번만 평가합니다.

In [14]:
test_loss, test_acc = evaluate(model, test_loader)
print(f"test_loss={test_loss:.4f}  test_acc={test_acc:.4f}")

# 모델과 토크나이저 저장 (Colab 런타임이 끊기면 사라지니, 필요하면 구글 드라이브에 저장하세요)
model.save_pretrained(cfg.save_dir)
tokenizer.save_pretrained(cfg.save_dir)

test_loss=0.2444  test_acc=0.8992


('nsmc-kcbert/tokenizer_config.json', 'nsmc-kcbert/tokenizer.json')

## 정리와 다음 단계

- 1 epoch만으로 test 정확도 89.92%를 얻었고, validation(90.16%)과 차이가 작아 과적합 징후는 보이지 않습니다.
- 개선해 볼 부분
  - `padding="max_length"` 대신 배치별 동적 패딩(`DataCollatorWithPadding`)으로 학습 속도 개선 (리뷰 평균 길이 35자로 패딩 낭비가 큼)
  - 중복 리뷰 제거(`drop_duplicates`) 후 성능 비교
  - epoch 수, 학습률 조정 실험과 `klue/roberta-base` 등 다른 사전학습 모델 비교
  - 오분류 사례 분석 (반어, 혼합 감성 리뷰)